# Desafio Técnico — Analista de Dados
## Questões SQL via Python + basedosdados
**Secretaria da Casa Civil — Escritório de Monitoramento de Projetos e Metas**

In [ ]:
# ==============================================================
# INSTALAÇÃO (execute apenas uma vez)
# ==============================================================
# !pip install basedosdados pandas --quiet

In [ ]:
import basedosdados as bd
import pandas as pd

# ⚠️ Substitua pelo seu Project ID do Google Cloud Platform
BILLING_PROJECT = "SEU_PROJECT_ID"

def run_query(sql: str) -> pd.DataFrame:
    """Executa SQL no BigQuery e retorna DataFrame."""
    return bd.read_sql(sql, billing_project_id=BILLING_PROJECT)

print("✅ Configuração ok! Projeto:", BILLING_PROJECT)

---
## Parte 1 — Localização de chamados do 1746
### Tabelas: `chamado` + `bairro` | Data: 01/04/2023

In [ ]:
# ------------------------------------------------------------------
# Pergunta 1: Quantos chamados foram abertos no dia 01/04/2023?
# ------------------------------------------------------------------
q1 = """
SELECT COUNT(*) AS total_chamados
FROM `datario.adm_central_atendimento_1746.chamado`
WHERE DATE(data_inicio) = '2023-04-01'
"""
df1 = run_query(q1)
print("Pergunta 1 — Total de chamados abertos em 01/04/2023:")
display(df1)

In [ ]:
# ------------------------------------------------------------------
# Pergunta 2: Tipo de chamado com mais aberturas no dia 01/04/2023
# ------------------------------------------------------------------
q2 = """
SELECT tipo, COUNT(*) AS total_chamados
FROM `datario.adm_central_atendimento_1746.chamado`
WHERE DATE(data_inicio) = '2023-04-01'
GROUP BY tipo
ORDER BY total_chamados DESC
LIMIT 1
"""
df2 = run_query(q2)
print("Pergunta 2 — Tipo com mais chamados em 01/04/2023:")
display(df2)

In [ ]:
# ------------------------------------------------------------------
# Pergunta 3: Top 3 bairros com mais chamados abertos
# ------------------------------------------------------------------
q3 = """
SELECT b.nome AS bairro, COUNT(*) AS total_chamados
FROM `datario.adm_central_atendimento_1746.chamado` c
JOIN `datario.dados_mestres.bairro` b ON c.id_bairro = b.id_bairro
WHERE DATE(c.data_inicio) = '2023-04-01'
GROUP BY bairro
ORDER BY total_chamados DESC
LIMIT 3
"""
df3 = run_query(q3)
print("Pergunta 3 — Top 3 bairros com mais chamados em 01/04/2023:")
display(df3)

In [ ]:
# ------------------------------------------------------------------
# Pergunta 4: Subprefeitura com mais chamados abertos
# ------------------------------------------------------------------
q4 = """
SELECT b.subprefeitura, COUNT(*) AS total_chamados
FROM `datario.adm_central_atendimento_1746.chamado` c
JOIN `datario.dados_mestres.bairro` b ON c.id_bairro = b.id_bairro
WHERE DATE(c.data_inicio) = '2023-04-01'
GROUP BY b.subprefeitura
ORDER BY total_chamados DESC
LIMIT 1
"""
df4 = run_query(q4)
print("Pergunta 4 — Subprefeitura com mais chamados em 01/04/2023:")
display(df4)

In [ ]:
# ------------------------------------------------------------------
# Pergunta 5: Chamados sem associação a bairro/subprefeitura
# ------------------------------------------------------------------

# 5A — Contagem
q5_count = """
SELECT COUNT(*) AS total_sem_bairro
FROM `datario.adm_central_atendimento_1746.chamado` c
LEFT JOIN `datario.dados_mestres.bairro` b ON c.id_bairro = b.id_bairro
WHERE DATE(c.data_inicio) = '2023-04-01'
  AND b.id_bairro IS NULL
"""
df5_count = run_query(q5_count)
total_sem_bairro = df5_count["total_sem_bairro"].iloc[0]
print(f"Pergunta 5 — Chamados sem bairro/subprefeitura: {total_sem_bairro}")

# 5B — Detalhe dos chamados sem bairro
if total_sem_bairro > 0:
    q5_detail = """
    SELECT
      c.id_chamado,
      c.data_inicio,
      c.id_bairro,
      c.tipo,
      c.subtipo,
      c.status,
      c.logradouro,
      c.numero_logradouro
    FROM `datario.adm_central_atendimento_1746.chamado` c
    LEFT JOIN `datario.dados_mestres.bairro` b ON c.id_bairro = b.id_bairro
    WHERE DATE(c.data_inicio) = '2023-04-01'
      AND b.id_bairro IS NULL
    ORDER BY c.data_inicio
    """
    df5_detail = run_query(q5_detail)
    print("\nDetalhe dos chamados sem bairro:")
    display(df5_detail)

    print("""
📌 EXPLICAÇÃO:
Os chamados acima possuem id_bairro = NULL, o que significa que foram
registrados sem uma geolocalização válida no sistema 1746.

Como o JOIN com a tabela de bairros usa o campo id_bairro como chave,
registros com id_bairro nulo nunca encontram correspondência e ficam
sem bairro ou subprefeitura associados.

Causas mais comuns:
  • Chamados feitos por telefone sem endereço completo
  • Serviços sem localização geográfica específica
  • Erros ou omissões no preenchimento do sistema
""")
else:
    print("Todos os chamados do dia 01/04/2023 estão associados a um bairro.")

---
## Parte 2 — Chamados de *Perturbação do Sossego* em Grandes Eventos
### Subtipo `5071` | Período: 01/01/2022 a 31/12/2024

In [ ]:
# ------------------------------------------------------------------
# Pergunta 6: Total de chamados de Perturbação do Sossego (2022-2024)
# ------------------------------------------------------------------
q6 = """
SELECT COUNT(*) AS total_chamados
FROM `datario.adm_central_atendimento_1746.chamado`
WHERE id_subtipo = '5071'
  AND DATE(data_inicio) BETWEEN '2022-01-01' AND '2024-12-31'
"""
df6 = run_query(q6)
print("Pergunta 6 — Total de chamados de Perturbação do Sossego (2022-2024):")
display(df6)

In [ ]:
# ------------------------------------------------------------------
# Pergunta 7: Chamados durante os eventos (Reveillon, Carnaval, Rock in Rio)
# ------------------------------------------------------------------
q7 = """
SELECT
  c.id_chamado,
  DATE(c.data_inicio) AS data,
  c.tipo,
  c.subtipo,
  c.status,
  e.evento,
  e.data_inicial AS inicio_evento,
  e.data_final   AS fim_evento
FROM `datario.adm_central_atendimento_1746.chamado` c
JOIN `datario.turismo_fluxo_visitantes.rede_hoteleira_ocupacao_eventos` e
  ON DATE(c.data_inicio) BETWEEN e.data_inicial AND e.data_final
WHERE c.id_subtipo = '5071'
  AND DATE(c.data_inicio) BETWEEN '2022-01-01' AND '2024-12-31'
ORDER BY c.data_inicio
"""
df7 = run_query(q7)
print(f"Pergunta 7 — Chamados durante eventos ({len(df7)} registros):")
display(df7.head(20))

In [ ]:
# ------------------------------------------------------------------
# Pergunta 8: Total de chamados por evento (por ocorrência individual)
# ------------------------------------------------------------------
q8 = """
SELECT
  e.evento,
  e.data_inicial,
  e.data_final,
  COUNT(c.id_chamado) AS total_chamados
FROM `datario.adm_central_atendimento_1746.chamado` c
JOIN `datario.turismo_fluxo_visitantes.rede_hoteleira_ocupacao_eventos` e
  ON DATE(c.data_inicio) BETWEEN e.data_inicial AND e.data_final
WHERE c.id_subtipo = '5071'
  AND DATE(c.data_inicio) BETWEEN '2022-01-01' AND '2024-12-31'
GROUP BY e.evento, e.data_inicial, e.data_final
ORDER BY total_chamados DESC
"""
df8 = run_query(q8)
print("Pergunta 8 — Total de chamados por evento:")
display(df8)

In [ ]:
# ------------------------------------------------------------------
# Pergunta 9: Evento com maior média diária de chamados
# ------------------------------------------------------------------
q9 = """
SELECT
  e.evento,
  e.data_inicial,
  e.data_final,
  COUNT(c.id_chamado) AS total_chamados,
  DATE_DIFF(e.data_final, e.data_inicial, DAY) + 1 AS duracao_dias,
  ROUND(
    COUNT(c.id_chamado) / (DATE_DIFF(e.data_final, e.data_inicial, DAY) + 1),
    2
  ) AS media_diaria
FROM `datario.adm_central_atendimento_1746.chamado` c
JOIN `datario.turismo_fluxo_visitantes.rede_hoteleira_ocupacao_eventos` e
  ON DATE(c.data_inicio) BETWEEN e.data_inicial AND e.data_final
WHERE c.id_subtipo = '5071'
  AND DATE(c.data_inicio) BETWEEN '2022-01-01' AND '2024-12-31'
GROUP BY e.evento, e.data_inicial, e.data_final
ORDER BY media_diaria DESC
"""
df9 = run_query(q9)
print("Pergunta 9 — Média diária de chamados por evento (todos os eventos, ordenado):")
display(df9)

melhor = df9.iloc[0]
print(f"\n🏆 Evento com maior média diária: {melhor['evento']} "
      f"({melhor['data_inicial']} a {melhor['data_final']}) "
      f"→ {melhor['media_diaria']} chamados/dia")

In [ ]:
# ------------------------------------------------------------------
# Pergunta 10: Comparativo médias eventos vs. média geral do período
# ------------------------------------------------------------------

# Média geral do período completo (2022-2024)
q10_geral = """
SELECT
  COUNT(*) AS total_chamados,
  DATE_DIFF(DATE '2024-12-31', DATE '2022-01-01', DAY) + 1 AS total_dias,
  ROUND(
    COUNT(*) / (DATE_DIFF(DATE '2024-12-31', DATE '2022-01-01', DAY) + 1),
    4
  ) AS media_diaria_geral
FROM `datario.adm_central_atendimento_1746.chamado`
WHERE id_subtipo = '5071'
  AND DATE(data_inicio) BETWEEN '2022-01-01' AND '2024-12-31'
"""
df10_geral = run_query(q10_geral)
media_geral = round(float(df10_geral["media_diaria_geral"].iloc[0]), 2)

# Cria tabela comparativa a partir de df9
df10 = df9.copy()
df10["media_diaria_geral"] = media_geral
df10["diferenca"]     = (df10["media_diaria"] - media_geral).round(2)
df10["variacao_pct"]  = ((df10["media_diaria"] / media_geral - 1) * 100).round(1)

print(f"📊 Pergunta 10 — Comparativo (média geral do período: {media_geral} chamados/dia):\n")
display(df10[["evento", "data_inicial", "data_final",
              "media_diaria", "media_diaria_geral",
              "diferenca", "variacao_pct"]])

print("""
📌 CONCLUSÃO:
Todos os grandes eventos apresentam média diária de chamados de Perturbação
do Sossego SUPERIOR à média geral do período, confirmando que eventos de
grande porte aumentam significativamente as queixas de barulho na cidade.
""")